# 01 — Data exploration and Gate 0

Before any training code exists, three assumptions the whole design rests on get checked.
Each one failed or surprised us at least once, so they are now assertions in `wcb.data`
rather than things we remember.

In [ ]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd, numpy as np
pd.set_option("display.width", 200)
from wcb.logging_utils import setup_logging
setup_logging(logging.WARNING)

In [ ]:
from wcb import config as C, data, features
frames = data.load_annotated(C.PRIMARY_SEED)
{s: len(df) for s, df in frames.items()}

## Gate 0.1 — are per-bank test splits equal?

**No.** The 70/15/15 split is global, not stratified by bank. Every bank contributes
exactly 1,000 sentences, but per-bank *test* counts range from 128 to 176.

This does not break the design. What the comparison needs is that the pooled and LOBO
models score the *same* items for a given bank, which filtering the aggregate still
guarantees.

In [ ]:
sizes = data.bank_test_sizes(frames)
print(f"min {sizes.min()} ({sizes.idxmin()})   max {sizes.max()} ({sizes.idxmax()})   total {sizes.sum()}")
sizes.to_frame("n_test").T

## Gate 0.2 — token lengths set the compute budget

`max_length` has to cover the bulk of the corpus; doubling it roughly doubles GPU time.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(C.MODEL_NAME)
allt = pd.concat(frames.values())[C.TEXT_COLUMN]
lens = np.array([len(tok(t)["input_ids"]) for t in allt])
print({f"p{p}": int(np.percentile(lens, p)) for p in (50, 90, 95, 99)}, "max", lens.max())
for L in (64, 128, 256):
    print(f"  max_length={L}: covers {100*(lens<=L).mean():.2f}%")

## Gate 0.3 — cross-split duplicates

Removed from **train and val only**. The test split is never modified: shrinking it
would break both the per-bank evaluation sets and comparability with the published WCB
numbers. `wcb.data.deduplicate` enforces this and asserts it.

## Label and bank distributions

Reproduces the proposal's figures: ~35% neutral / 33% dovish / 28% hawkish / 3%
irrelevant, and a very uneven hawkish share across banks (China ~9%, Turkey ~45%).

In [ ]:
combined = pd.concat(frames.values())
print((combined[C.STANCE_COLUMN].value_counts(normalize=True) * 100).round(1).to_string())
hawk = (combined.groupby(C.BANK_COLUMN)[C.STANCE_COLUMN]
        .apply(lambda s: (s == "hawkish").mean() * 100).sort_values())
print("\nhawkish share by bank (%):")
print(hawk.round(1).to_string())

## Pre-registered bank features

Fixed *before* any gap is computed — see `wcb.features`.

In [ ]:
feat = features.compute_bank_features(frames, include_embeddings=False).dropna(axis=1, how="all")
feat.sort_values("stance_js_divergence", ascending=False).round(4)